# Capítulo 8 · Llamadas a funciones

Tres herramientas para el asistente de Ciclos Norte (estado de un pedido, existencias y coste de envío), descritas con esquemas JSON, y el ciclo completo: el modelo pide una herramienta, el programa la ejecuta y el modelo responde con el resultado. Se mide con veinte peticiones y cuatro variantes del mensaje de sistema y de las descripciones.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo y los datos

In [ ]:
"""Capítulo 8: llamadas a funciones. Modelo abierto, en GPU o CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen2.5-1.5B-Instruct"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-0.6B"             # sin GPU: más pequeño y más rápido
MODELO = None                              # None: elegir según el entorno
DATOS = Path("datos")
REPOSITORIO = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/"



def cargar(nombre=None):
    """Carga el modelo y su tokenizador en la GPU si la hay; si no, en la CPU."""
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    if nombre is None:
        nombre = MODELO_GPU if dispositivo == "cuda" else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(nombre, dtype=torch.float32)
    modelo.to(dispositivo).eval()
    print(f"Modelo {nombre} en {dispositivo}")
    return tok, modelo


def descargar(archivo):
    """Un archivo JSON de la carpeta datos; en Colab, se descarga."""
    ruta = DATOS / archivo
    if not ruta.exists():
        DATOS.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORIO + ruta.as_posix(), ruta)
    return json.loads(ruta.read_text(encoding="utf-8"))

In [ ]:
tok, modelo = cargar(MODELO)
PEDIDOS, ALMACEN = descargar("pedidos.json"), descargar("existencias.json")
print(len(PEDIDOS), "pedidos y", len(ALMACEN), "productos en el almacén")

## 2. Las herramientas: funciones normales de Python

Se pueden probar sin ningún modelo.

In [ ]:
def sin_tildes(texto):
    """Minúsculas y sin tildes."""
    return "".join(c for c in unicodedata.normalize("NFD", str(texto).lower())
                   if unicodedata.category(c) != "Mn").strip()


def estado_pedido(numero):
    """Estado de un pedido de la base de datos."""
    cifras = re.sub(r"\D", "", str(numero))
    pedido = PEDIDOS.get(f"CN-{cifras}")
    if pedido is None:
        return {"error": f"No existe ningún pedido con el número {numero}."}
    return {"numero": f"CN-{cifras}", **pedido}


def existencias(producto, talla=None):
    """Unidades en el almacén de un producto, por nombre o por referencia."""
    buscado = sin_tildes(producto)
    for ref, datos in ALMACEN.items():
        if buscado in (sin_tildes(ref), sin_tildes(datos["nombre"])):
            unidades = datos["unidades"]
            if talla is not None:
                unidades = {str(talla): unidades.get(str(talla).upper(), 0)}
            return {"referencia": ref, "nombre": datos["nombre"],
                    "unidades": unidades,
                    "reposicion": datos.get("reposicion", "sin fecha prevista")}
    return {"error": f"No hay datos de existencias de «{producto}»."}


ZONAS = {"baleares": ["baleares", "mallorca", "menorca", "ibiza", "formentera",
                      "palma"],
         "canarias": ["canarias", "tenerife", "gran canaria", "lanzarote",
                      "fuerteventura", "la palma", "las palmas"],
         "portugal": ["portugal", "lisboa", "oporto"], "andorra": ["andorra"],
         "extranjero": ["francia", "italia", "alemania", "reino unido",
                        "estados unidos", "mexico", "paris", "londres", "roma"]}


def coste_envio(destino, importe=0):
    """Coste y plazo de envío de un pedido según el destino y su importe."""
    lugar = sin_tildes(destino)
    zona = next((z for z, lugares in ZONAS.items()
                 if any(l in lugar for l in lugares)), "península")
    if zona == "extranjero":
        return {"error": f"No hacemos envíos a {destino}. Se puede pedir "
                         "presupuesto en hola@ciclosnorte.es."}
    if zona == "península":
        coste = 0 if float(importe or 0) > 50 else 4.95
        return {"zona": zona, "coste_euros": coste, "plazo": "48-72 horas"}
    tarifas = {"baleares": (9.95, "3-5 días laborables"),
               "canarias": (14.95, "5-7 días laborables"),
               "portugal": (12.95, "3-5 días laborables"),
               "andorra": (12.95, "3-5 días laborables")}
    coste, plazo = tarifas[zona]
    return {"zona": zona, "coste_euros": coste, "plazo": plazo}


FUNCIONES = {"estado_pedido": estado_pedido, "existencias": existencias,
             "coste_envio": coste_envio}

In [ ]:
print(estado_pedido("cn 10871"))
print(existencias("Aero 3", "54"))
print(coste_envio("Mallorca", 80))
print(coste_envio("Francia"))

## 3. Describir las herramientas al modelo

In [ ]:
def herramienta(nombre, descripcion, parametros, obligatorios):
    """Esquema de una herramienta en el formato que entienden los modelos."""
    return {"type": "function", "function": {
        "name": nombre, "description": descripcion,
        "parameters": {"type": "object", "properties": parametros,
                       "required": obligatorios}}}


ESQUEMAS = [
    herramienta(
        "estado_pedido",
        "Consulta el estado de un pedido de la tienda: si está en preparación, "
        "enviado, entregado, retrasado o cancelado, con sus fechas y el número "
        "de seguimiento. Úsala solo si el cliente da su número de pedido.",
        {"numero": {"type": "string",
                    "description": "Número de pedido, con el formato CN-12345."}},
        ["numero"]),
    herramienta(
        "existencias",
        "Consulta cuántas unidades de un producto quedan en el almacén, por "
        "talla, y la fecha de reposición si está agotado.",
        {"producto": {"type": "string",
                      "description": "Nombre del producto (por ejemplo, Aero 3) "
                                     "o su referencia (por ejemplo, CR-7010)."},
         "talla": {"type": "string",
                   "description": "Talla, si el cliente la indica (S, M, L, 54…)."}},
        ["producto"]),
    herramienta(
        "coste_envio",
        "Calcula el coste y el plazo de envío de un pedido según el destino y "
        "el importe del pedido.",
        {"destino": {"type": "string",
                     "description": "Ciudad, provincia, isla o país de destino."},
         "importe": {"type": "number",
                     "description": "Importe del pedido en euros; 0 si no se sabe."}},
        ["destino"]),
]


# Las mismas herramientas, sin descripciones: solo nombres y tipos.
ESQUEMAS_POBRES = [
    herramienta(e["function"]["name"], "",
                {p: {"type": d["type"]}
                 for p, d in e["function"]["parameters"]["properties"].items()},
                e["function"]["parameters"]["required"]) for e in ESQUEMAS]


SISTEMA_ES = (
    "Eres el asistente de atención al cliente de Ciclos Norte, una tienda de "
    "bicicletas de Oviedo. Responde en español, en una o dos frases. Para los "
    "datos de pedidos, existencias y envíos, usa las herramientas; no los "
    "inventes. Si te falta un dato para usar una herramienta, pídeselo al "
    "cliente. Otros datos de la tienda: hay 30 días para devolver un producto; "
    "la garantía es de 2 años; la tienda abre de lunes a viernes de 10:00 a "
    "20:00 y los sábados de 10:00 a 14:00.")


# Las mismas instrucciones, en inglés: el idioma en el que estos modelos han
# aprendido sobre todo a usar herramientas.
SISTEMA_EN = (
    "You are the customer service assistant of Ciclos Norte, a bicycle shop in "
    "Oviedo (Spain). Always answer in Spanish, in one or two sentences. Use the "
    "tools for orders, stock and shipping; never invent those data. If a tool "
    "needs data the customer has not given, ask for it. Other shop data: 30 days "
    "to return a product; 2-year warranty; open Monday to Friday 10:00-20:00 and "
    "Saturdays 10:00-14:00.")

In [ ]:
print(json.dumps(ESQUEMAS[0], ensure_ascii=False, indent=1))

## 4. El ciclo

In [ ]:
def generar(mensajes, esquemas, max_tokens=200):
    """Una respuesta del modelo, con las herramientas descritas en la plantilla."""
    texto = tok.apply_chat_template(mensajes, tools=esquemas, tokenize=False,
                                    add_generation_prompt=True,
                                    enable_thinking=False)
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    respuesta = tok.decode(salida[0, entrada["input_ids"].shape[1]:],
                           skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", respuesta, flags=re.S).strip()


def leer_llamadas(respuesta):
    """Las llamadas que pide el modelo, escritas entre <tool_call> y </tool_call>."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)</tool_call>", respuesta, re.S):
        try:
            llamada = json.loads(bloque)
            llamadas.append({"name": llamada["name"],
                             "arguments": llamada.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            llamadas.append({"name": None, "arguments": {}, "texto": bloque})
    return llamadas


def ejecutar(llamada):
    """Ejecuta una llamada después de comprobar el nombre y los argumentos."""
    funcion = FUNCIONES.get(llamada["name"])
    if funcion is None:
        return {"error": f"No existe la herramienta {llamada['name']}."}
    try:
        return funcion(**llamada["arguments"])
    except (TypeError, ValueError) as e:
        return {"error": f"Argumentos no válidos: {e}"}


def atender(peticion, esquemas=ESQUEMAS, sistema=SISTEMA_EN, max_pasos=3):
    """Atiende una petición: el modelo puede pedir herramientas hasta
    max_pasos veces antes de dar la respuesta final."""
    mensajes = ([{"role": "system", "content": sistema}] if sistema else []) + [
        {"role": "user", "content": peticion}]
    registro = []
    for _ in range(max_pasos):
        respuesta = generar(mensajes, esquemas)
        llamadas = leer_llamadas(respuesta)
        if not llamadas:
            return respuesta, registro
        mensajes.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": l} for l in llamadas if l["name"]]})
        for llamada in llamadas:
            resultado = ejecutar(llamada)
            registro.append({**llamada, "resultado": resultado})
            mensajes.append({"role": "tool", "name": llamada["name"],
                             "content": json.dumps(resultado, ensure_ascii=False)})
    return "No he podido completar la consulta; paso tu mensaje al equipo.", registro

Una petición con cada mensaje de sistema. Fíjate en si el modelo llama a la herramienta:

In [ ]:
peticion = "¿Dónde está mi pedido CN-10871?"
for nombre, sistema in (("Español", SISTEMA_ES), ("Inglés", SISTEMA_EN)):
    respuesta, registro = atender(peticion, ESQUEMAS, sistema)
    print(nombre, "→ llamadas:", [(r["name"], r["arguments"]) for r in registro])
    print("   ", respuesta)

## 5. Veinte peticiones y cuatro variantes

In [ ]:
def numero(esperado):
    """Comprueba el número de pedido, con o sin el prefijo CN-."""
    return lambda a: re.sub(r"\D", "", str(a.get("numero", ""))) == esperado[3:]


def producto(esperado, talla=None):
    """Comprueba el producto (nombre o referencia) y, si hace falta, la talla."""
    return lambda a: (sin_tildes(a.get("producto", "")) in
                      (sin_tildes(esperado[0]), sin_tildes(esperado[1]))
                      and (talla is None or str(a.get("talla", "")).upper() == talla))


def destino(lugar):
    """Comprueba que el destino contiene el lugar esperado."""
    return lambda a: sin_tildes(lugar) in sin_tildes(a.get("destino", ""))


# Petición, herramienta esperada (o None), comprobación de los argumentos y
# textos de los que alguno debe aparecer en la respuesta final.
PETICIONES = [
    ("¿Dónde está mi pedido CN-10871?", "estado_pedido", numero("CN-10871"),
     ["enviado", "ER-558201"]),
    ("Hola, soy Óscar, pedido 11033. ¿Cuándo me llega?", "estado_pedido",
     numero("CN-11033"), ["13"]),
    ("¿Ha salido ya el pedido cn-11107?", "estado_pedido", numero("CN-11107"),
     ["enviado", "ha salido"]),
    ("Mi pedido CN-11302 se está retrasando, ¿qué pasa?", "estado_pedido",
     numero("CN-11302"), ["talla L", "existencias", "15"]),
    ("¿Cuándo me devolvéis el dinero del pedido CN-10990?", "estado_pedido",
     numero("CN-10990"), ["14"]),
    ("¿Qué pasa con mi pedido CN-99999?", "estado_pedido", numero("CN-99999"),
     ["no existe", "no encontr", "no hay ningún", "no aparece"]),
    ("¿Dónde está mi pedido?", None, None, ["número"]),
    ("¿Tenéis la Aero 3 en talla 54?", "existencias",
     producto(("Aero 3", "CR-7010"), "54"), ["20 de octubre", "2026-10-20", "20/10"]),
    ("¿Queda alguna Urbana 5 en talla M?", "existencias",
     producto(("Urbana 5", "UR-5150"), "M"), ["3", "tres"]),
    ("¿Hay existencias de la referencia EL-9050?", "existencias",
     producto(("E-Urbana 600", "EL-9050")), ["noviembre", "2026-11-05", "5/11"]),
    ("¿Tenéis el Casco Peque en talla S?", "existencias",
     producto(("Casco Peque", "CA-4150"), "S"), ["no", "agotad"]),
    ("¿Cuántas E-Plegable os quedan?", "existencias",
     producto(("E-Plegable", "EL-9200")), ["3", "tres"]),
    ("¿Tenéis la Fixie Norte?", "existencias",
     producto(("Fixie Norte", "UR-5470")), ["no ", "consult"]),
    ("¿Cuánto me cuesta enviar un pedido de 30 euros a Sevilla?", "coste_envio",
     destino("Sevilla"), ["4,95", "4.95"]),
    ("Si compro una bici de 600 euros, ¿me la enviáis gratis a Zaragoza?",
     "coste_envio", destino("Zaragoza"), ["gratis", "gratuito", "sin coste"]),
    ("¿Cuánto cuesta enviar a Mallorca un pedido de 80 euros?", "coste_envio",
     destino("Mallorca"), ["9,95", "9.95"]),
    ("¿Enviáis a Francia?", "coste_envio", destino("Francia"),
     ["presupuesto", "no "]),
    ("¿Cuánto tiempo tengo para devolver un producto?", None, None, ["30 días"]),
    ("Hola, buenos días.", None, None, None),
    ("¿Abrís los sábados?", None, None, ["10:00", "14:00"]),
]

In [ ]:
def evaluar(esquemas=ESQUEMAS, sistema=SISTEMA_EN):
    """Atiende las veinte peticiones y comprueba herramienta, argumentos y
    respuesta."""
    filas = []
    for peticion, esperada, argumentos, claves in PETICIONES:
        respuesta, registro = atender(peticion, esquemas, sistema)
        usadas = [r["name"] for r in registro]
        primera = registro[0] if registro else None
        fila = {"peticion": peticion, "respuesta": respuesta, "registro": registro,
                "herramienta": (usadas[:1] == [esperada]) if esperada else not usadas}
        fila["argumentos"] = (fila["herramienta"] and esperada is not None
                              and argumentos(primera["arguments"]))
        fila["respuesta_bien"] = claves is None or any(
            c.lower() in respuesta.lower() for c in claves)
        filas.append(fila)
    return filas


def resumen(filas):
    """Herramientas bien elegidas (de 20), argumentos correctos (de 16),
    respuestas correctas (de 20) y número de llamadas."""
    con = [f for f, p in zip(filas, PETICIONES) if p[1]]
    return {"herramienta": sum(f["herramienta"] for f in filas),
            "argumentos": sum(f["argumentos"] for f in con),
            "respuesta": sum(f["respuesta_bien"] for f in filas),
            "llamadas": sum(len(f["registro"]) for f in filas)}


VARIANTES = {
    "Sistema en español": (ESQUEMAS, SISTEMA_ES),
    "Sin mensaje de sistema": (ESQUEMAS, None),
    "Sistema en inglés": (ESQUEMAS, SISTEMA_EN),
    "Sistema en inglés, sin descripciones": (ESQUEMAS_POBRES, SISTEMA_EN),
}

In [ ]:
resultados = {}
for nombre, (esquemas, sistema) in VARIANTES.items():
    resultados[nombre] = evaluar(esquemas, sistema)
    print(f"{nombre:38s}", resumen(resultados[nombre]))

Las peticiones de la mejor variante, con sus llamadas y una marca en las que fallan:

In [ ]:
for f in resultados["Sistema en inglés"]:
    marca = "  " if f["herramienta"] and f["respuesta_bien"] else "✗ "
    print(marca + f["peticion"])
    print("      llamadas:", [(r["name"], r["arguments"]) for r in f["registro"]])
    print("      respuesta:", f["respuesta"][:150])

## Ejercicios

1. Explica con tus palabras quién hace qué en una llamada a una función: el modelo, la plantilla de chat y el programa.

2. Imprime el texto completo que recibe el modelo, con `apply_chat_template` y las herramientas, para una de las peticiones. ¿Dónde aparecen las descripciones? ¿Qué instrucciones añade la plantilla?

3. Para las cinco peticiones en las que el modelo no usa la herramienta con el mensaje en inglés, prueba a mejorar la descripción de la herramienta correspondiente. ¿Consigues que la use sin estropear otras?

4. Añade una cuarta herramienta, `buscar_producto(descripcion)`, que devuelva los productos del catálogo cuya descripción contenga alguna palabra de la consulta. ¿Cambia la elección de las otras tres? ¿Se confunde el modelo?

5. Cambia `estado_pedido` para que solo acepte números con el formato exacto `CN-12345` y devuelva un error si no. ¿Cuántas peticiones fallan ahora? ¿Corrige el modelo el número después del error?

6. Escribe una petición que necesite dos herramientas, como «¿Tenéis la Urbana 5 en talla M y cuánto cuesta enviarla a Mallorca?». ¿Llama el modelo a las dos? ¿En una sola respuesta o en dos pasos?

7. Haz que la herramienta de existencias devuelva un resultado más explícito, por ejemplo `"3 unidades"` en lugar de `3`. ¿Se corrige el error de la E-Plegable?

8. Traduce también las descripciones de las herramientas al inglés y repite la prueba con los dos modelos. ¿Mejora? ¿Y si traduces solo el mensaje de sistema para el modelo pequeño?

9. Añade una herramienta peligrosa, `cancelar_pedido(numero)`, y modifica `ejecutar` para que, antes de ejecutarla, pida una confirmación por teclado con `input`. Prueba con la petición «Cancela mi pedido CN-11033».

10. **Asistente completo.** Combina el RAG del capítulo 6 con las herramientas de este capítulo: añade una herramienta `buscar_documentos(consulta)` que devuelva los tres fragmentos más relevantes. Pruébalo con diez peticiones mezcladas y escribe un breve informe con lo que funciona y lo que no.

## Referencias

Modelos [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) y [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B). [Toolformer](https://arxiv.org/abs/2302.04761) (Schick y colaboradores, 2023). [Model Context Protocol](https://modelcontextprotocol.io). Lección 11.09 de [AI Engineering from Scratch](https://github.com/rohitg00/ai-engineering-from-scratch). La tienda Ciclos Norte, sus pedidos y su almacén son ficticios. Código y experimentos propios.